# BEA 2-way generator

An attempt at tidying the code up at least a little so that eg. Aisling has a fighting chance of seeing what it's even doing.

**Convert the raw cells to Python to check the functions are working OK**

The core aspect is the call to `amati_bea.induce_theory`, and we can then write the theory to a file. The rules will come from `clingo_program_files/bea_rules.amati`.

There's a number of legacy imports here.

In [1]:
import amati_bea as amati

In [2]:
import pandas as pd

from sklearn.metrics import f1_score

In [3]:
from operator import itemgetter
import pickle
import datetime
import os

In [4]:
THEORY_ROOT_DIR='/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318'

## Import the data

We've got the DataFrames containing the data in the file `bea_2way_data.pickle`. Load that in:

In [5]:
with open("bea_2way_data.pickle", "br") as fIn:
    d = pickle.load(fIn)

all_questions_df = d["questions"]
all_responses_df = d["responses"]
all_text_df = d["text"]

In [6]:
all_questions_df.head()

,original_question_id,question_id,question,correct,incorrect_1,incorrect_2,correct_id,incorrect_1_id,incorrect_2_id
0,fcf5ff56-5e98-4eb7-8df3-afb0544d9739,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,"Die SuS erklären, dass durch die unterschiedli...","Die SuS erklären nicht, dass dadurch die Mögli...","Die SuS erklären, dass durch die unterschiedli...",c_00,i_00,j_00
1,c51b89a6-0219-4eee-bb89-b2b5168998f5,q_01,Begründe deine Auswahl.,Die Schüler:innen stellen einen Zusammenhang z...,Die Schüler:innen stellen keinen Zusammenhang ...,Die Schüler:innen stellen einen Zusammenhang z...,c_01,i_01,j_01
2,379ba896-bb8f-440d-9d32-5dc9ca0297fe,q_02,Formuliere selbst eine Fragestellung.,Die SuS beschreiben die durchschnittliche Gesc...,Die SuS beschreiben weder die durchschnittlich...,Die SuS beschreiben entweder die durchschnittl...,c_02,i_02,j_02
3,cb99dfb3-2788-4a10-b934-7ea8276b2b44,q_03,Beschreibe auf Basis Deines Vorwissens die Erk...,"Die Schüler:innen erläutern den Vorteil, dass ...",Die Schüler:innen erläutern den erweiterten Bl...,"Die Schüler:innen erläuten den Vorteil, dass d...",c_03,i_03,j_03
4,b85b1a95-35ba-473e-a164-4602a4a13a2f,q_04,In dem Video siehst du die abstrakte Funktions...,Die SuS beschreiben die Funktionsweise eines D...,Die SuS beschreiben die Funktionsweise nicht k...,"Die SuS beschreiben nur einen Teil, entweder d...",c_04,i_04,j_04


In [7]:
all_responses_df.head()

,original_response_id,response_id,question_id,response,score,partition
0,e78122dd-9f3c-456e-a546-fc905b023807,r_0111,q_00,Im Winter ist weniger Strahlung weil die Sonne...,incorrect,train
1,4cf98df5-8d79-4144-aeea-37df3bec5de5,r_0115,q_00,Es kann im Süden und Sommer mehr elektrische E...,incorrect,train
2,6550b62c-cc65-458d-a609-a7a38c45cbca,r_0129,q_00,da im sommer mehr die somme scheint gibt es au...,incorrect,train
3,e4d9c6e1-ed55-468a-9101-89218a989852,r_0178,q_00,im Sommer gewinnt man die meiste Energie wobei...,incorrect,train
4,7d3e3263-668f-4283-ab26-edd6d4c43be1,r_0292,q_00,Im Winter ist am wenigsten Energie da im Winte...,incorrect,train


In [8]:
all_text_df.head()

,id,text,i,word_text,lemma
0,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,0,Erläutere,erläuter
1,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,1,die,der
2,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,2,Auswirkungen,auswirkung
3,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,3,der,der
4,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,4,jahreszeitabhängigen,jahreszeitabhängig


## Run on a single example case as a tester

I realised that my code doesn't always assign the same Amati codes to the original IDs, so start by selecting a test case from the original question code:

Let's try working through the functions in `amati_bea.py`.

In [9]:
TEST_QUESTION_ID = "fcf5ff56-5e98-4eb7-8df3-afb0544d9739"
TEST_QUESTION_ID

'fcf5ff56-5e98-4eb7-8df3-afb0544d9739'

We now pull out the Series containing this ID as the one to work with:

In [10]:
question_ss = all_questions_df.set_index("original_question_id").T[TEST_QUESTION_ID]

question_ss

question_id                                                    q_00
question          Erläutere die Auswirkungen der jahreszeitabhän...
correct           Die SuS erklären, dass durch die unterschiedli...
incorrect_1       Die SuS erklären nicht, dass dadurch die Mögli...
incorrect_2       Die SuS erklären, dass durch die unterschiedli...
correct_id                                                     c_00
incorrect_1_id                                                 i_00
incorrect_2_id                                                 j_00
Name: fcf5ff56-5e98-4eb7-8df3-afb0544d9739, dtype: object

For the responses, want just the training cases, not the trial cases.

In [11]:
responses_df = all_responses_df.query("`question_id`==@question_ss['question_id']").query(
    "`partition`=='train'"
)
responses_df

,original_response_id,response_id,question_id,response,score,partition
0,e78122dd-9f3c-456e-a546-fc905b023807,r_0111,q_00,Im Winter ist weniger Strahlung weil die Sonne...,incorrect,train
1,4cf98df5-8d79-4144-aeea-37df3bec5de5,r_0115,q_00,Es kann im Süden und Sommer mehr elektrische E...,incorrect,train
2,6550b62c-cc65-458d-a609-a7a38c45cbca,r_0129,q_00,da im sommer mehr die somme scheint gibt es au...,incorrect,train
3,e4d9c6e1-ed55-468a-9101-89218a989852,r_0178,q_00,im Sommer gewinnt man die meiste Energie wobei...,incorrect,train
4,7d3e3263-668f-4283-ab26-edd6d4c43be1,r_0292,q_00,Im Winter ist am wenigsten Energie da im Winte...,incorrect,train
...,...,...,...,...,...,...
92,8ceef6e1-5902-485c-8b2d-23204dcc3f4d,r_7672,q_00,"Im Winter bin ich mir sicher , dass die Strahl...",incorrect,train
93,7df30d21-fc6c-4577-a14d-e621068f6847,r_7675,q_00,"Im Sommer scheint die Sonne sehr viel , aber i...",correct,train
94,14645435-9103-45a8-89b2-60e2ebf4502e,r_7698,q_00,Spannung und Stromstärke in Abhängigkeit der W...,incorrect,train
95,9410add7-f86b-4d73-8334-785dbd5cf22b,r_7739,q_00,Man kann erkennen das man im Winter nicht die ...,incorrect,train


In [12]:
texts_used = [
    question_ss["question_id"],
    question_ss["correct_id"],
    question_ss["incorrect_1_id"],
    question_ss["incorrect_2_id"],
] + list(responses_df["response_id"])

text_df = all_text_df.query("`id`.isin(@texts_used)")[["id", "i", "lemma"]]
text_df

,id,i,lemma
0,q_00,0,erläuter
1,q_00,1,der
2,q_00,2,auswirkung
3,q_00,3,der
4,q_00,4,jahreszeitabhängig
...,...,...,...
10524,r_7739,30,immer
10525,r_7739,31,mehr
10526,r_7739,32,strom
10527,r_7739,33,benötigen


OK, that should cover what we want.

----------------

## Generate the positive and negative cases

OK, now to start with the `amati_bea` functions.

Let's start off by generating positive and negative cases.

In [13]:
accuracy_classes = set(responses_df["score"])
accuracy_classes

{'correct', 'incorrect'}

In [14]:
responses_df

,original_response_id,response_id,question_id,response,score,partition
0,e78122dd-9f3c-456e-a546-fc905b023807,r_0111,q_00,Im Winter ist weniger Strahlung weil die Sonne...,incorrect,train
1,4cf98df5-8d79-4144-aeea-37df3bec5de5,r_0115,q_00,Es kann im Süden und Sommer mehr elektrische E...,incorrect,train
2,6550b62c-cc65-458d-a609-a7a38c45cbca,r_0129,q_00,da im sommer mehr die somme scheint gibt es au...,incorrect,train
3,e4d9c6e1-ed55-468a-9101-89218a989852,r_0178,q_00,im Sommer gewinnt man die meiste Energie wobei...,incorrect,train
4,7d3e3263-668f-4283-ab26-edd6d4c43be1,r_0292,q_00,Im Winter ist am wenigsten Energie da im Winte...,incorrect,train
...,...,...,...,...,...,...
92,8ceef6e1-5902-485c-8b2d-23204dcc3f4d,r_7672,q_00,"Im Winter bin ich mir sicher , dass die Strahl...",incorrect,train
93,7df30d21-fc6c-4577-a14d-e621068f6847,r_7675,q_00,"Im Sommer scheint die Sonne sehr viel , aber i...",correct,train
94,14645435-9103-45a8-89b2-60e2ebf4502e,r_7698,q_00,Spannung und Stromstärke in Abhängigkeit der W...,incorrect,train
95,9410add7-f86b-4d73-8334-785dbd5cf22b,r_7739,q_00,Man kann erkennen das man im Winter nicht die ...,incorrect,train


In [15]:
pos_cases_df = responses_df[["response_id", "question_id", "score"]].sort_values(
    "response_id", ignore_index=True
)

pos_cases_df.sort_values("response_id", ignore_index=True)

,response_id,question_id,score
0,r_0111,q_00,incorrect
1,r_0115,q_00,incorrect
2,r_0129,q_00,incorrect
3,r_0178,q_00,incorrect
4,r_0292,q_00,incorrect
...,...,...,...
92,r_7672,q_00,incorrect
93,r_7675,q_00,correct
94,r_7698,q_00,incorrect
95,r_7739,q_00,incorrect


In [16]:
neg_cases_df = (
    pd.concat([responses_df.assign(classes=c) for c in accuracy_classes])
    .query("`score`!=`classes`")
    .drop(["score", "partition"], axis="columns")
    .rename({"classes": "score"}, axis="columns")[
        ["response_id", "question_id", "score"]
    ]
    .sort_values("response_id", ignore_index=True)
)


neg_cases_df

,response_id,question_id,score
0,r_0111,q_00,correct
1,r_0115,q_00,correct
2,r_0129,q_00,correct
3,r_0178,q_00,correct
4,r_0292,q_00,correct
...,...,...,...
92,r_7672,q_00,correct
93,r_7675,q_00,incorrect
94,r_7698,q_00,correct
95,r_7739,q_00,correct


------------------------

I think what I want is to be able to send those structures to `amati_bea.py` and see how it goes from there.

Check that the function `build_amati_training_file` works correctly:

In [17]:
# default file name and stopwords

amati.build_amati_training_file(
    positive_examples_df=pos_cases_df,
    negative_examples_df=neg_cases_df,
    responses_df=responses_df,
    text_df=text_df,
    rulesfile="clingo_program_files/bea_rules.amati",
    # filename="amati_induction.lp",
    question_ss=question_ss,
    stopwords=True,
)

True

OK, that seems OK. Now, what about an actual induction run? Check the behaviour of `induce_once` first:

In [18]:
x = amati.induce_once(
    positive_examples_df=pos_cases_df,
    negative_examples_df=neg_cases_df,
    responses_df=responses_df,
    text_df=text_df,
    question_ss=question_ss,
    rulesfile="clingo_program_files/bea_rules.amati",
    time_limit=10,
    min_precision=0.8,
    min_coverage=2,
)

x

{'result': 'SATISFIABLE',
 'selected_rule': ('rule1', 'selected_rule(rule1)'),
 'predicted_grade': ('incorrect', 'predicted_grade(incorrect)'),
 'parameters': [(1, '"es"', 'parameter(1,"es")'),
  (2, '"sommer"', 'parameter(2,"sommer")')],
 'positive_covered': ['r_0115',
  'r_0129',
  'r_0178',
  'r_0292',
  'r_0792',
  'r_1140',
  'r_1210',
  'r_2196',
  'r_2876',
  'r_3295',
  'r_3882',
  'r_4025',
  'r_4218',
  'r_5282',
  'r_5809',
  'r_5846',
  'r_5895',
  'r_6185',
  'r_7672'],
 'negative_covered': ['r_1461', 'r_2270', 'r_7675'],
 'evaluations': {'precision': 86,
  'compression': 16,
  'coverage': 16,
  'accuracy': 86},
 'positive_df':    response_id question_id      score  covered
 0       r_0111        q_00  incorrect    False
 1       r_0115        q_00  incorrect     True
 2       r_0129        q_00  incorrect     True
 3       r_0178        q_00  incorrect     True
 4       r_0292        q_00  incorrect     True
 ..         ...         ...        ...      ...
 92      r_7672 

OK, and then do a complete run:

In [19]:
x = amati.induce_theory(
    positive_examples_df=pos_cases_df,
    negative_examples_df=neg_cases_df,
    responses_df=responses_df,
    text_df=text_df,
    question_ss=question_ss,
    rulesfile="clingo_program_files/bea_rules.amati",
    time_limit=10,
    min_precision=0.8,
    min_coverage=2,
)

97 84 75 69 64 54 49 45 42 39 36 34 32 28 26 24 22 20 18 16 

In [20]:
x

[{'result': 'SATISFIABLE',
  'selected_rule': ('rule1', 'selected_rule(rule1)'),
  'predicted_grade': ('incorrect', 'predicted_grade(incorrect)'),
  'parameters': [(1, '"sonne"', 'parameter(1,"sonne")'),
   (2, '"seehr"', 'parameter(2,"seehr")')],
  'positive_covered': ['r_0111',
   'r_0792',
   'r_0958',
   'r_1903',
   'r_1942',
   'r_2238',
   'r_2863',
   'r_4025',
   'r_4179',
   'r_4218',
   'r_4655',
   'r_5809',
   'r_6546'],
  'negative_covered': ['r_2270', 'r_5376', 'r_7675'],
  'evaluations': {'precision': 81,
   'compression': 10,
   'coverage': 10,
   'accuracy': 81},
  'positive_df':    response_id question_id      score  covered
  0       r_0111        q_00  incorrect     True
  1       r_0115        q_00  incorrect    False
  2       r_0129        q_00  incorrect    False
  3       r_0178        q_00  incorrect    False
  4       r_0292        q_00  incorrect    False
  ..         ...         ...        ...      ...
  92      r_7672        q_00  incorrect    False
  93 

In [21]:
x[-1]

{'result': 'UNSATISFIABLE',
 'selected_rule': None,
 'predicted_grade': None,
 'parameters': None,
 'positive_covered': [],
 'negative_covered': [],
 'evaluation': None,
 'positive_df':    response_id question_id      score  covered
 11      r_0869        q_00  incorrect    False
 12      r_0895        q_00  incorrect    False
 16      r_1081        q_00    correct    False
 20      r_1461        q_00    correct    False
 23      r_1628        q_00  incorrect    False
 35      r_2504        q_00  incorrect    False
 37      r_2728        q_00  incorrect    False
 45      r_3501        q_00  incorrect    False
 49      r_3843        q_00  incorrect    False
 60      r_5064        q_00    correct    False
 68      r_5665        q_00    correct    False
 78      r_6130        q_00    correct    False
 79      r_6185        q_00  incorrect    False
 81      r_6576        q_00    correct    False
 85      r_6774        q_00  incorrect    False
 95      r_7739        q_00  incorrect    False

Actually, let's just call that the theory. Should be able to pickle it...

In [22]:
def induce_theory(
    original_question_id,
    data_file="bea_2way_data.pickle",
    rulesfile="clingo_program_files/bea_rules.amati",
    time_limit=120,
    min_precision=0.8,
    min_coverage=2,
):
    """question_id is the original version, not the"
    shortened version"""

    with open("bea_2way_data.pickle", "br") as fIn:
        d = pickle.load(fIn)
        all_questions_df = d["questions"]
        all_responses_df = d["responses"]
        all_text_df = d["text"]

    question_ss = all_questions_df.set_index("original_question_id").T[
        original_question_id
    ]
    question_id = question_ss["question_id"]

    # To induce the theory, just want the training cases

    responses_df = all_responses_df.query("`question_id`==@question_id").query(
        "`partition`=='train'"
    )

    texts_used = [
        question_ss["question_id"],
        question_ss["correct_id"],
        question_ss["incorrect_1_id"],
        question_ss["incorrect_2_id"],
    ] + list(responses_df["response_id"])

    text_df = all_text_df.query("`id`.isin(@texts_used)")[["id", "i", "lemma"]]

    # Next, need the positive and negative cases

    accuracy_classes = set(responses_df["score"])

    pos_cases_df = responses_df[["response_id", "question_id", "score"]].sort_values(
        "response_id", ignore_index=True
    )

    neg_cases_df = (
        pd.concat([responses_df.assign(classes=c) for c in accuracy_classes])
        .query("`score`!=`classes`")
        .drop(["score", "partition"], axis="columns")
        .rename({"classes": "score"}, axis="columns")[
            ["response_id", "question_id", "score"]
        ]
        .sort_values("response_id", ignore_index=True)
    )

    output = {
        "theory": amati.induce_theory(
            positive_examples_df=pos_cases_df,
            negative_examples_df=neg_cases_df,
            responses_df=responses_df,
            text_df=text_df,
            question_ss=question_ss,
            rulesfile=rulesfile,
            time_limit=time_limit,
            min_precision=min_precision,
            min_coverage=min_coverage,
        )
    }

    output["original_question_id"] = original_question_id
    output["question_id"] = question_id

    output["all_questions_df"] = all_questions_df
    output["all_responses_df"] = all_responses_df
    output["all_text_df"] = all_text_df

    with open(rulesfile) as fIn:
        output["rules"] = fIn.read()

    return output

## Apply to the complete set

We should now be able to apply to the complete set of questions. 

In [23]:
with open("bea_2way_data.pickle", "br") as fIn:
    p = pickle.load(fIn)

for q_id in set(p["questions"]["original_question_id"]):
    print(q_id)
    d = datetime.datetime.now()
    theory_file_name = os.path.join(THEORY_ROOT_DIR, f"theory_{q_id}_{d.date()}_{d.time()}.pickle"
    )
    with open(theory_file_name, "wb") as fOut:
        pickle.dump(induce_theory(q_id), fOut)

64636904-aab6-4e10-a4dc-39638117731a
216 24 19 17 15 8e686080-6db7-473b-baa0-6f17412fae5c
157 116 93 74 64 55 47 41 35 31 29 27 25 23 21 19 dee4136e-e922-4b83-8033-24350ea1610c
124 50 40 36 33 31 28 26 24 22 20 dddf3a9d-2f40-402c-b650-3db1b27128ca
74 63 55 48 44 41 39 37 35 33 31 29 26 24 22 b5fb6924-1da3-45b1-a386-68e95d80b057
196 93 79 69 65 60 54 50 47 45 43 41 39 36 34 31 29 27 25 23 33c179c8-ee50-4a7b-b139-8b2984356400
45 29 25 21 18 16 14 278f349f-1552-41c4-bec2-1de41838f229
145 57 39 31 26 23 21 19 16 14 12 8a35dfaf-e9d0-466d-98b8-1e1b132bb450
187 161 139 127 118 112 107 103 100 97 94 90 86 84 81 79 76 72 70 68 66 64 62 60 58 56 54 52 50 48 46 44 42 40 38 d9e775ff-b54e-4558-86eb-0251737ecdf1
27 14 7 5 3 d14cc74c-805a-45f7-9ad3-2e9f19395ca0
73 30 23 20 17 15 12 10 8 0f036c26-4a47-48ba-b913-f27487286ac7
64 55 49 45 41 38 34 32 30 8144573c-97d6-484a-b68b-c16b7602fa7b
98 81 68 61 55 51 46 42 39 37 35 32 30 26 379ba896-bb8f-440d-9d32-5dc9ca0297fe
180 23 17 15 13 11 48071573-9e44-4ae9